In [ ]:
import sqlite3

db_path = "pizza_pizza.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("Database connected.")

Database connected.


In [ ]:
cursor.execute("""
CREATE TABLE Customer(
    member_id TEXT NOT NULL,
    name TEXT NOT NULL,
    points_balance INTEGER NOT NULL,
    CONSTRAINT PK_Customer PRIMARY KEY(member_id),
    CONSTRAINT CK_Customer_points_balance CHECK(points_balance >= 0)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE FranchiseLocation(
    store_id TEXT PRIMARY KEY,
    street_address TEXT NOT NULL,
    city TEXT NOT NULL,
    province TEXT NOT NULL,
    postal_code TEXT NOT NULL,
    store_phone TEXT NOT NULL,
    seating_capacity INTEGER NOT NULL CHECK(seating_capacity > 0)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE MenuItem(
    item_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    base_price REAL NOT NULL CHECK(base_price > 0),
    category TEXT NOT NULL CHECK(category IN ('Pizza', 'Drink', 'Side'))
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE Employee(
    employee_id TEXT PRIMARY KEY,
    store_id TEXT NOT NULL,
    name TEXT NOT NULL,
    hourly_rate REAL NOT NULL CHECK(hourly_rate >= 0),
    FOREIGN KEY (store_id) REFERENCES FranchiseLocation(store_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE CustomerPhone(
    member_id TEXT NOT NULL,
    phone_number TEXT NOT NULL,
    PRIMARY KEY(member_id, phone_number),
    FOREIGN KEY (member_id) REFERENCES Customer(member_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE Driver(
    employee_id TEXT PRIMARY KEY,
    vehicle_type TEXT NOT NULL,
    insurance_expiry TEXT NOT NULL,
    FOREIGN KEY (employee_id) REFERENCES Employee(employee_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE KitchenStaff(
    employee_id TEXT PRIMARY KEY,
    food_handler_cert_id TEXT NOT NULL,
    FOREIGN KEY (employee_id) REFERENCES Employee(employee_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE StoreManager(
    manager_id TEXT PRIMARY KEY,
    store_id TEXT NOT NULL UNIQUE,
    name TEXT NOT NULL,
    start_date TEXT NOT NULL,
    FOREIGN KEY (store_id) REFERENCES FranchiseLocation(store_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE SizeVariant(
    variant_id TEXT PRIMARY KEY,
    item_id TEXT NOT NULL,
    size_name TEXT NOT NULL,
    crust_type TEXT NOT NULL,
    FOREIGN KEY (item_id) REFERENCES MenuItem(item_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE CustomerOrder(
    order_id TEXT PRIMARY KEY,
    member_id TEXT NOT NULL,
    order_date TEXT NOT NULL,
    order_type TEXT NOT NULL CHECK(order_type IN ('Delivery', 'Pickup', 'Walk-in')),
    FOREIGN KEY (member_id) REFERENCES Customer(member_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE Delivery(
    order_id TEXT PRIMARY KEY,
    driver_id TEXT NOT NULL,
    delivery_time TEXT NOT NULL,
    tip_amount REAL NOT NULL CHECK(tip_amount >= 0),
    FOREIGN KEY (order_id) REFERENCES CustomerOrder(order_id),
    FOREIGN KEY (driver_id) REFERENCES Driver(employee_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE OrderItem(
    order_item_id TEXT PRIMARY KEY,
    order_id TEXT NOT NULL,
    item_id TEXT NOT NULL,
    variant_id TEXT NOT NULL,
    quantity INTEGER NOT NULL CHECK(quantity >= 1),
    unit_price REAL NOT NULL CHECK(unit_price > 0),
    FOREIGN KEY (order_id) REFERENCES CustomerOrder(order_id),
    FOREIGN KEY (item_id) REFERENCES MenuItem(item_id),
    FOREIGN KEY (variant_id) REFERENCES SizeVariant(variant_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE PizzaCustomization(
    order_item_id TEXT NOT NULL,
    custom_id TEXT NOT NULL,
    instruction TEXT NOT NULL,
    extra_charge REAL NOT NULL CHECK(extra_charge >= 0),
    PRIMARY KEY(order_item_id, custom_id),
    FOREIGN KEY (order_item_id) REFERENCES OrderItem(order_item_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE RawMaterial(
    material_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    unit TEXT NOT NULL,
    expiry_date TEXT NOT NULL,
    quantity_in_stock INTEGER NOT NULL CHECK(quantity_in_stock >= 0),
    reorder_level INTEGER NOT NULL CHECK(reorder_level >= 0),
    store_id TEXT NOT NULL,
    FOREIGN KEY (store_id) REFERENCES FranchiseLocation(store_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE RecipeConsistsOf(
    item_id TEXT NOT NULL,
    material_id TEXT NOT NULL,
    quantity_required REAL NOT NULL CHECK(quantity_required > 0),
    PRIMARY KEY(item_id, material_id),
    FOREIGN KEY (item_id) REFERENCES MenuItem(item_id),
    FOREIGN KEY (material_id) REFERENCES RawMaterial(material_id)
);
""")

In [ ]:
cursor.execute("""
CREATE TABLE MenuItemAllergen(
    item_id TEXT NOT NULL,
    allergen TEXT NOT NULL,
    PRIMARY KEY(item_id, allergen),
    FOREIGN KEY (item_id) REFERENCES MenuItem(item_id)
);
""")
conn.commit()